<a href="https://colab.research.google.com/github/Avishek2020/Agentic_AI_Projects/blob/main/Rulebased_Multi_tool_system_AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###  Libraries

In [1]:
!pip install ddgs faiss-cpu sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.0/67.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 58.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 81.4 MB/s eta 0:00:00


### Step 2 — Define Tools - 🔎 Web Search Tool

In [2]:
from ddgs import DDGS

def search_tool(query):
    with DDGS() as ddgs:
        results = [r["body"] for r in ddgs.text(query, max_results=3)]
    return "\n".join(results)

### 🧮 Calculator Tool


```
eval() is a Python function that executes a string as a Python expression and returns its result.
```



In [3]:
import re

def calculator_tool(query):
    try:
        # Extract numbers
        numbers = re.findall(r"\d+", query)

        if len(numbers) < 2:
            return "Need at least two numbers"

        A = int(numbers[0])
        B = int(numbers[1])

        # Decide operation (default = addition)
        if "add" in query or "+" in query:
            return str(A + B)
        elif "subtract" in query or "-" in query:
            return str(A - B)
        elif "multiply" in query or "*" in query:
            return str(A * B)
        elif "divide" in query or "/" in query:
            return str(A / B)
        else:
            return str(A + B)  # default

    except:
        return "Error in calculation"

### 🧠 Step 3 — Memory (Vector DB)

In [4]:
from sentence_transformers import SentenceTransformer # SentenceTransformer → converts text → vectors (embeddings)
import faiss # fast similarity search (Facebook library)
import numpy as np

# Loading embedding model, This model converts text into 384-dimensional vectors
model = SentenceTransformer('all-MiniLM-L6-v2')

memory = [] # memory → stores original text
vectors = faiss.IndexFlatL2(384)  # vectors → stores embeddings (384 = vector size (must match model))
# IndexFlatL2: Uses L2 distance (Euclidean) to measure similarity

def add_memory(text):
    emb = model.encode([text])
    vectors.add(np.array(emb).astype('float32')) # Add embedding to FAISS index
    memory.append(text) # Store original text in list

def retrieve_memory(query):
    if len(memory) == 0:
        return ""
    emb = model.encode([query])
    D, I = vectors.search(np.array(emb).astype('float32'), k=2) # Search top 2 most similar memories - D = distances (how close) - I = indices of best matches
    return "\n".join([memory[i] for i in I[0]])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 🧭 Step 4 — Planner (VERY IMPORTANT)

This is where your agent becomes “smart”.
👉 This is rule-based for now

In [5]:
def planner(query):
    q = query.lower()

    if "calculate" in q or any(char.isdigit() for char in q):
        return "calculator"
    elif "latest" in q or "news" in q:
        return "search"
    else:
        return "general"

### ⚙️ Step 5 — Agent Executor

In [42]:
def agent(query):
    print(f"\n🧑 User: {query}")

    # Step 1: Check memory
    mem = retrieve_memory(query)

    # Step 2: Plan
    tool = planner(query)
    print("🧭 Planner chose:", tool)

    # Step 3: Execute tool
    if tool == "search":
        tool_output = search_tool(query)
    elif tool == "calculator":
        tool_output = calculator_tool(query)
    else:
        tool_output = "General response (no tool used)"

    # Step 4: Combine
    final = (
            f"Old Memory:\n{mem}\n"
           # f"Tool Output:\n{tool_output}\n"
            f"Final Answer :\t{tool_output}"
        )

        # Step 5: Save ONLY clean memory (IMPORTANT FIX)
    add_memory(f"{query} -> {tool_output}")

    return final

### 🧪 Step 6 — Test

In [48]:
print(agent("latest AI news"))
print(agent("calculate 6 * 4"))
print(agent("what did I ask before?"))


🧑 User: latest AI news
🧭 Planner chose: search
Old Memory:
what did I ask before? -> General response (no tool used)
what did I ask before? -> General response (no tool used)
Final Answer :	2 days ago - AI News delivers the latest updates in artificial intelligence, machine learning, deep learning, enterprise AI, and emerging tech worldwide.
2 days ago - News coverage on artificial intelligence and machine learning tech, the companies building them, and the ethical issues AI raises today.
3 days ago - The latest news and top stories on artificial intelligence, including AI chatbots like Microsoft’s ChatGPT, Apple’s AI Chatbot and Google’s Bard.

🧑 User: calculate 6 * 4
🧭 Planner chose: calculator
Old Memory:
calculate 6 * 4 -> 24
calculate 6 * 4 -> 24
Final Answer :	24

🧑 User: what did I ask before?
🧭 Planner chose: general
Old Memory:
what did I ask before? -> General response (no tool used)
what did I ask before? -> General response (no tool used)
Final Answer :	General response (n

In [59]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['Copy of Rulebased_Multi-tool system_AI_Agent.ipynb',
 'Rulebased_Multi-tool_system_AI_Agent.ipynb',
 'Copy of Rulebased_Multi-tool_system_AI_Agent.ipynb']

In [61]:
import nbformat

file = "/content/drive/MyDrive/Colab Notebooks/Rulebased_Multi-tool_system_AI_Agent.ipynb"

nb = nbformat.read(file, as_version=4)

# 🔥 REMOVE widget metadata (this is the root problem)
nb.metadata.pop("widgets", None)

# Clean outputs (recommended for GitHub)
for cell in nb.cells:
    cell["outputs"] = []
    cell["execution_count"] = None

# Save new file
nbformat.write(nb, "/content/drive/MyDrive/Colab Notebooks/Rulebased_Multi-tool_system_AI_Agent.ipynb")

print("✅ Done: clean.ipynb created")

✅ Done: clean.ipynb created
